# Three of Kairui Hao's figures, redrawn

**Shows:** three separate figures on synthetic data (no inference is run).

1. `fig_6_3` (thesis Fig. 6.3): convergence of two variational guides and two samplers. The top row is
   the whole run; the bottom row enlarges its start.
2. `fig_14` (neural information field filter paper, Fig. 14): posterior against measurement for four
   outputs. Left column: the whole record. Right column: a short window of it.
3. `fig_16` (same paper, Fig. 16): posterior predictive against measurement for four states and their accelerations.

**Graph types:** convergence trace with a zoomed row (`lf.ribbon`, `lf.paths`, `lf.zoom`), and a dense
signal under a band from draws (`lf.ribbon`, `lf.signal`).

**Writes:** `fig_6_3.svg`, `fig_14.svg` and `fig_16.svg` next to this notebook.

**To tweak:** change the capitalised settings at the top of a **Figure** cell and re-run that cell.
Every figure has its own **Data** cell just above it.

In [ ]:
import numpy as np

import labfigs as lf

lf.use()
rng = np.random.default_rng(3)

## Synthetic signals

Helpers that only make up data for the three figures. With real results, drop this cell and load arrays
of the shapes listed in each **Data** cell.

In [ ]:
steps = np.linspace(0, 2e6, 4001)


def wander(scale, chains=5, keep=0.98):
    noise = np.zeros((chains, steps.size))
    for i in range(1, steps.size):
        noise[:, i] = keep * noise[:, i - 1] + scale * rng.normal(size=chains)
    return noise


def sweep(t, envelope, noise):
    clean = envelope * np.sin(2 * np.pi * (4 * t + 0.008 * t**2))
    draws = clean * (1 + 0.04 * rng.normal(size=(200, 1))) + 0.02 * envelope.max() * rng.normal(size=(200, t.size))
    return draws, 1.05 * clean + noise * (envelope + 0.03 * envelope.max()) * rng.normal(size=t.size)


def ramp_up(t, start, peak, power):
    return np.clip((t - start) / (peak - start), 0, 1) ** power * np.clip(1 - (t - peak) / 6, 0, 1) ** 2


def diamond(t, centre, reach, power):
    return np.clip(1 - np.abs(t - centre) / reach, 0, 1) ** power

## Data for Fig. 6.3

- `steps`: iteration numbers, length 4001 (from the cell above)
- `early`: the part of `steps` that the bottom row enlarges
- `columns`: one entry per parameter, with
  - `truth`: its true value
  - `guides`: `{name: (estimate, half-width)}`, each an array over `steps`; the names become legend entries
  - `chains`: `{name: chains}`, each of shape `(5, len(steps))`

In [ ]:
early = slice(0, 201)
columns = []
for truth in (0.3, -1.0, 1.0):
    start = rng.uniform(-0.4, 0.5)
    guides = {
        name: (truth + (start - truth) * np.exp(-steps / rate), 0.02 + 0.5 * np.exp(-2 * steps / rate))
        for name, rate in (("Non-relaxed SVI", 2.5e4), ("Relaxed SVI", 1.2e4))
    }
    chains = {
        name: truth + (start + rng.normal(0, 0.3, (5, 1)) - truth) * np.exp(-steps / rate) + wander(scale)
        for name, rate, scale in (("Relaxed SGLD", 2e5, 0.004), ("Relaxed preconditioned SGLD", 3e4, 0.008))
    }
    columns.append({"truth": truth, "guides": guides, "chains": chains})

## Figure 1: Fig. 6.3

Settings at the top of the cell:

- `NAME`: file name of the figure, without `.svg`
- `PANEL_ASPECT`: height of one panel divided by its width
- `CHAIN_COLOURS`, `CHAIN_ALPHA`, `CHAIN_WIDTH`: colour of each sampler, and the opacity and line width of its chains
- `TOP_UNIT`, `TOP_TICKS`, `TOP_TICK_LABELS`: iterations per unit of the top row's x axis, and its ticks
- `BOTTOM_UNIT`, `BOTTOM_TICKS`, `BOTTOM_TICK_LABELS`: the same for the bottom row
- `ZOOM_WINDOW`: the stretch of the top x axis, in its own units, that the bottom row enlarges
- `X_LABEL`: the shared x label
- `Y_LIMITS`, `Y_TICKS`: the y axis of every panel; the y label is built from the column number
- `LEGEND_COLUMNS`: number of columns in the legend above the figure

In [ ]:
NAME = "fig_6_3"
PANEL_ASPECT = 0.8
CHAIN_COLOURS, CHAIN_ALPHA, CHAIN_WIDTH = ("C2", "C3"), 0.6, 0.5
TOP_UNIT, TOP_TICKS, TOP_TICK_LABELS = 1e6, [0, 1, 2], ["0", "$10^6$", r"$2\times10^6$"]
BOTTOM_UNIT, BOTTOM_TICKS, BOTTOM_TICK_LABELS = 1e5, [0, 0.5, 1], ["0", r"$5\times10^4$", "$10^5$"]
ZOOM_WINDOW = (0, 0.1)
X_LABEL = "Iterations"
Y_LIMITS, Y_TICKS = (-1.3, 1.3), [-1, 0, 1]
LEGEND_COLUMNS = 3

fig, axes = lf.subplots(2, len(columns), aspect=PANEL_ASPECT, sharey=True)
for j, column in enumerate(columns):
    for ax, part, unit in ((axes[0, j], slice(None), TOP_UNIT), (axes[1, j], early, BOTTOM_UNIT)):
        x = steps[part] / unit
        for name, (mid, width) in column["guides"].items():
            lf.ribbon(ax, x, (mid - width)[part], mid[part], (mid + width)[part], label=name)
        for (name, chain), color in zip(column["chains"].items(), CHAIN_COLOURS):
            lf.paths(ax, x, chain[:, part], color=color, alpha=CHAIN_ALPHA, lw=CHAIN_WIDTH, label=name)
        lf.truth(ax, column["truth"])
        ax.set(yticks=Y_TICKS, ylim=Y_LIMITS, ylabel=rf"$k_{j + 1}/\bar{{k}}_{j + 1}$")
    axes[0, j].set(xticks=TOP_TICKS, xticklabels=TOP_TICK_LABELS)
    axes[1, j].set(xticks=BOTTOM_TICKS, xticklabels=BOTTOM_TICK_LABELS)
    lf.zoom(axes[0, j], axes[1, j], *ZOOM_WINDOW)
fig.supxlabel(X_LABEL)
lf.legend_above(fig, ncols=LEGEND_COLUMNS)
lf.save(fig, NAME)

## Data for Fig. 14

- `seconds_14`: time axis, length 9000
- `outputs`: one entry per row, each `(draws, measured)` with `draws` of shape `(200, len(seconds_14))`
  and `measured` of the same length as the time axis

In [ ]:
seconds_14 = np.arange(0, 90, 0.01)
outputs = [
    sweep(seconds_14, envelope, noise)
    for envelope, noise in (
        (0.038 * ramp_up(seconds_14, 20, 80, 0.6), 0.02),
        (50 * ramp_up(seconds_14, 20, 80, 2.5), 0.08),
        (0.038 * diamond(seconds_14, 45, 38, 0.7), 0.02),
        (50 * diamond(seconds_14, 45, 35, 2.5), 0.08),
    )
]

## Figure 2: Fig. 14

Settings at the top of the cell:

- `NAME`: file name of the figure, without `.svg`
- `PANEL_ASPECT`: height of one panel divided by its width
- `ROWS`: one entry per row: its y label, its y tick (ticks sit at minus that, 0 and plus that),
  and the start and end of the window shown in the right column
- `POSTERIOR_LABEL`, `POSTERIOR_COLOUR`: legend entry and colour of the band
- `MEASUREMENT_LABEL`: legend entry of the measured signal
- `X_LABEL`, `WIDE_TICKS`, `WINDOW_TICK_STEP`: the shared x label, the x ticks of the left column,
  and the spacing of the x ticks in the right column

In [ ]:
NAME = "fig_14"
PANEL_ASPECT = 0.36
ROWS = (
    (r"$y_1$ (m)", 0.025, (75, 85)),
    (r"$y_2$ (m/s$^2$)", 50, (75, 85)),
    (r"$y_3$ (m)", 0.025, (40, 50)),
    (r"$y_4$ (m/s$^2$)", 50, (40, 50)),
)
POSTERIOR_LABEL, POSTERIOR_COLOUR = "NSVI posterior", lf.CRIMSON
MEASUREMENT_LABEL = "Measurement"
X_LABEL, WIDE_TICKS, WINDOW_TICK_STEP = "$t$ (s)", np.arange(0, 91, 30), 2

fig, axes = lf.subplots(len(ROWS), 2, aspect=PANEL_ASPECT, sharey="row")
for (wide, detail), (label, tick, (lo, hi)), (draws, measured) in zip(axes, ROWS, outputs):
    for ax, keep in ((wide, slice(None)), (detail, (seconds_14 >= lo) & (seconds_14 <= hi))):
        lf.ribbon(ax, seconds_14[keep], draws[:, keep], color=POSTERIOR_COLOUR, lw=0.3, label=POSTERIOR_LABEL)
        lf.signal(ax, seconds_14[keep], measured[keep], label=MEASUREMENT_LABEL)
    wide.set(xticks=WIDE_TICKS, yticks=[-tick, 0, tick], ylabel=label)
    detail.set(xticks=np.arange(lo, hi + 1, WINDOW_TICK_STEP))
fig.supxlabel(X_LABEL)
lf.legend_above(fig)
lf.save(fig, NAME)

## Data for Fig. 16

- `seconds_16`: time axis, length 9300
- `states`: one entry per row; each row holds two panels (a state and its acceleration), and each panel
  is `(draws, measured, scale)` with `draws` of shape `(200, len(seconds_16))`, `measured` of the same
  length as the time axis, and `scale` the size of the signal, used for the y ticks

In [ ]:
seconds_16 = np.arange(5, 98, 0.01)
shaker = np.convolve(rng.normal(size=seconds_16.size), np.ones(25) / 25, "same") * ((seconds_16 > 19) & (seconds_16 < 80))
jump = np.where(seconds_16 < 42, 0.3 * ramp_up(seconds_16, 15, 42, 2),
                (1 - 0.3 * (seconds_16 - 42) / 46) * np.clip((92 - seconds_16) / 4, 0, 1))
states = []
for envelope, scales in (
    (None, (0.005, 0.5)),
    (jump, (0.025, 20)),
    (ramp_up(seconds_16, 28, 85, 0.7), (0.025, 25)),
    (diamond(seconds_16, 57, 40, 0.7), (0.025, 25)),
):
    pair = []
    for scale, power in zip(scales, (1, 2.5)):
        if envelope is None:
            clean = 1.5 * scale * shaker
            draws = clean + 0.03 * scale * rng.normal(size=(200, seconds_16.size))
            measured = 1.2 * clean + 0.03 * scale * rng.normal(size=seconds_16.size)
        else:
            draws, measured = sweep(seconds_16, 1.1 * scale * envelope**power, 0.05)
        pair.append((draws, measured, scale))
    states.append(pair)

## Figure 3: Fig. 16

Settings at the top of the cell:

- `NAME`: file name of the figure, without `.svg`
- `PANEL_ASPECT`: height of one panel divided by its width
- `LABELS`: one entry per row: the y labels of its left and right panel
- `POSTERIOR_LABEL`, `POSTERIOR_COLOUR`: legend entry and colour of the band
- `MEASUREMENT_LABEL`: legend entry of the measured signal
- `X_LABEL`, `X_TICKS`: the shared x axis

The y ticks of each panel sit at minus, 0 and plus the `scale` stored with its data.

In [ ]:
NAME = "fig_16"
PANEL_ASPECT = 0.36
LABELS = (
    (r"$x_1$ (m)", r"$a_1$ (m/s$^2$)"),
    (r"$x_2$ (m)", r"$a_2$ (m/s$^2$)"),
    (r"$x_3$ (m)", r"$a_3$ (m/s$^2$)"),
    (r"$x_4$ (m)", r"$a_4$ (m/s$^2$)"),
)
POSTERIOR_LABEL, POSTERIOR_COLOUR = "NSVI posterior predictive", lf.CRIMSON
MEASUREMENT_LABEL = "Measurement"
X_LABEL, X_TICKS = "$t$ (s)", np.arange(20, 101, 20)

fig, axes = lf.subplots(len(LABELS), 2, aspect=PANEL_ASPECT, sharex=True)
for row, labels, pair in zip(axes, LABELS, states):
    for ax, label, (draws, measured, scale) in zip(row, labels, pair):
        lf.ribbon(ax, seconds_16, draws, color=POSTERIOR_COLOUR, lw=0.3, label=POSTERIOR_LABEL)
        lf.signal(ax, seconds_16, measured, label=MEASUREMENT_LABEL)
        ax.set(xticks=X_TICKS, yticks=[-scale, 0, scale], ylabel=label)
fig.supxlabel(X_LABEL)
lf.legend_above(fig)
lf.save(fig, NAME)